# Basic Portfolio Analysis Notebook

Are you an investor looking to gain deeper insights into your portfolio's performance? Perhaps you're curious about how different assets contribute to your overall returns, or you want to simulate future performance to make more informed decisions.

## Install Required Packages

First, ensure you have the required packages installed. Run the following command in a Jupyter notebook cell:

In [ ]:
%pip install yfinance numpy matplotlib -q

## Import Necessary Libraries

Import the necessary libraries for our analysis.

In [ ]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from datetime import datetime, timedelta


In [ ]:
# Create your portfolio here
tickers = ['SPY','QQQ']
weights = [0.60,0.40]
portfolio = dict(zip(tickers, weights))
print('Portfolio:', portfolio)

# Define the start date and end date
start_date = '2000-01-01'
end_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
print('Start Date:', start_date)
print('End Date  :', end_date)

# Define the risk-free rate:
risk_free_rate = 0.02
print('Risk-free Rate:', risk_free_rate)

# Investment amounts for Monte Carlo simulation
initial_investment = 10000  # Initial portfolio value in dollars
monthly_investment = 400    # Monthly contribution in dollars
print('Initial Investment:', initial_investment)
print('Monthly Investment:', monthly_investment)

## DataLoader Class

Create a class to fetch and preprocess financial data.

In [ ]:
class DataLoader:
    def __init__(self, tickers, start_date, end_date):
        self.tickers = tickers
        self.start_date = start_date
        self.end_date = end_date

    def fetch_data(self):
        data = yf.download(self.tickers, start=self.start_date, end=self.end_date, auto_adjust=True)['Close']
        return data

## PerformanceMetrics Class

Create a class to calculate various performance metrics.

In [ ]:
class PerformanceMetrics:
    @staticmethod
    def calculate_annual_return(data):
        # Lisää .ffill() ennen pct_change()
        returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        # Käytetään geometrista laskentaa jokaiselle tickerille
        total_growth = (1 + returns).prod()
        n_years = len(returns) / 252
        return total_growth**(1/n_years) - 1

    @staticmethod
    def calculate_annual_volatility(data):
        annual_volatility = data.pct_change(fill_method=None).dropna(how='all').fillna(0).std() * (252 ** 0.5)
        return annual_volatility

    @staticmethod
    def calculate_sharpe_ratio(data, risk_free_rate=risk_free_rate):
        annual_return = PerformanceMetrics.calculate_annual_return(data)
        annual_volatility = PerformanceMetrics.calculate_annual_volatility(data)
        sharpe_ratio = (annual_return - risk_free_rate) / annual_volatility
        return sharpe_ratio

    @staticmethod
    def calculate_max_drawdown(data):
        cumulative_returns = (1 + data.pct_change(fill_method=None)).cumprod()
        peak = cumulative_returns.expanding(min_periods=1).max()
        drawdown = (cumulative_returns / peak) - 1
        max_drawdown = drawdown.min()
        return max_drawdown

    @staticmethod
    def calculate_sortino_ratio(data, risk_free_rate=risk_free_rate):
        returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        # Nollataan positiiviset tuotot
        downside_returns = np.where(returns < 0, returns, 0)
        # Lasketaan RMS (root-mean-square) keskihajonta
        downside_deviation = np.sqrt(np.mean(downside_returns**2)) * (252 ** 0.5)
        annual_return = PerformanceMetrics.calculate_annual_return(data)
        sortino_ratio = (annual_return - risk_free_rate) / downside_deviation
        return sortino_ratio

    @staticmethod
    def calculate_var(data, confidence_level=0.95):
        returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        var = np.percentile(returns, (1 - confidence_level) * 100)
        return var

## PortfolioAnalysis Class

Create a class to perform portfolio analysis.

## PortfolioVisualization Class

Create a class to visualize portfolio performance, including plotting the total portfolio return over time.

In [ ]:
class PortfolioVisualization:
    @staticmethod
    def plot_performance(data):
        # 1. Lasketaan kumulatiivinen tulos (sisältää NaN-arvoja katkoissa)
        returns = data.pct_change(fill_method=None)
        cumulative_returns = (1 + returns).cumprod()
        
        # 2. TÄMÄ ON RATKAISU: Lineaarinen interpolaatio
        # Se täyttää tyhjät kohdat laskemalla suoran linjan arvojen välille
        clean_cumulative = cumulative_returns.interpolate(method='linear')
        
        # 3. Piirretään kaavio tästä interpoloidusta datasta
        ax = clean_cumulative.plot(figsize=(10, 6))
        plt.title('Portfolio Component Cumulative Returns')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Returns')
        plt.grid(True)
        plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        plt.show()

    @staticmethod
    def plot_portfolio_return(data, weights):
        returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        weighted_returns = returns.dot(weights)
        cumulative_portfolio_returns = (1 + weighted_returns).cumprod()
        cumulative_portfolio_returns.plot(figsize=(10, 6))
        plt.title('Total Portfolio Cumulative Return')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Returns')
        plt.grid(True)
        plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        plt.show()

## BenchmarkComparison Class

Compare your portfolio's performance against common market benchmarks. Calculate alpha, beta, tracking error, and information ratio.

## Example Usage: Compute Portfolio Returns

Now, let's use the classes we've created to analyze a sample portfolio.

In [ ]:
# Load data
data_loader = DataLoader(tickers, start_date, end_date)
data = data_loader.fetch_data()

# Calculate performance metrics
annual_return = PerformanceMetrics.calculate_annual_return(data)
annual_volatility = PerformanceMetrics.calculate_annual_volatility(data)
sharpe_ratio = PerformanceMetrics.calculate_sharpe_ratio(data)
max_drawdown = PerformanceMetrics.calculate_max_drawdown(data)
sortino_ratio = PerformanceMetrics.calculate_sortino_ratio(data)
var = PerformanceMetrics.calculate_var(data)

print('\n********** Individual Ticker Stats **********')
print("\nAnnual Return (%):", np.round(100*annual_return, 2))
print("\nAnnual Volatility (%):", np.round(100*annual_volatility, 2))
print("\nSharpe Ratio:", np.round(sharpe_ratio, 2))
print("\nMax Drawdown (%):", np.round(100*max_drawdown, 2))
print("\nSortino Ratio:", np.round(sortino_ratio, 2))
print("\nValue at Risk (VaR):", np.round(var, 2))

# Perform portfolio analysis
portfolio_analysis = PortfolioAnalysis(data, weights)
portfolio_return = portfolio_analysis.calculate_portfolio_return()
portfolio_volatility = portfolio_analysis.calculate_portfolio_volatility()
portfolio_sharpe_ratio = portfolio_analysis.calculate_portfolio_sharpe_ratio()

print('\n********** Portfolio Return Stats **********')
print("Portfolio Return (%):", np.round(100*portfolio_return, 2))
print("Portfolio Volatility (%):", np.round(100*portfolio_volatility, 2))
print("Portfolio Sharpe Ratio:", np.round(portfolio_sharpe_ratio, 2))

## Visualize Portfolio Performance

In [ ]:
# Visualize portfolio performance
PortfolioVisualization.plot_performance(data)

In [ ]:
# Visualize total portfolio return
PortfolioVisualization.plot_portfolio_return(data, weights)

## Benchmark Comparison

Compare your portfolio against a market benchmark like the S&P 500 (SPY).

In [ ]:
# Compare portfolio against S&P 500 benchmark
benchmark = BenchmarkComparison(data, weights, benchmark_ticker='SPY', risk_free_rate=risk_free_rate)
benchmark.generate_report()
benchmark.plot_cumulative_returns()

## Monte Carlo Simulation

The Monte Carlo simulation projects potential future portfolio values by generating thousands of random return scenarios based on historical return distributions and asset correlations.

**Key Features:**
- Uses portfolio weights to calculate weighted returns
- Tracks cumulative portfolio value over time
- Provides percentile-based statistics (5th, 25th, 50th, 75th, 95th)
- Visualizes uncertainty with percentile bands
- Calculates probability of loss


### MonteCarloSimulation Class

Create a class to perform Monte Carlo simulations.

In [ ]:
class MonteCarloSimulation:
    """
    Monte Carlo simulation for portfolio performance projection.
    
    Simulates multiple future paths for a portfolio based on historical
    return distributions, accounting for asset correlations.
    """
    
    def __init__(self, data, weights, num_simulations=1000, time_horizon=252, initial_investment=10000):
        """
        Initialize Monte Carlo simulation.
        
        Parameters:
        -----------
        data : pd.DataFrame
            Historical price data for portfolio assets
        weights : array-like
            Portfolio weights (must sum to 1.0)
        num_simulations : int
            Number of simulation paths to generate
        time_horizon : int
            Number of trading days to simulate (252 = 1 year)
        initial_investment : float
            Starting portfolio value
        """
        self.data = data
        self.weights = np.array(weights)
        self.num_simulations = num_simulations
        self.time_horizon = time_horizon
        self.initial_investment = initial_investment
        
        # Validate weights
        assert len(self.weights) == len(data.columns), "Weights must match number of assets"
        assert np.isclose(sum(self.weights), 1.0), "Weights must sum to 1.0"
        
        # Store simulation results
        self._results = None

    def simulate(self):
        """
        Run Monte Carlo simulation.
        
        Returns:
        --------
        np.ndarray
            Array of shape (num_simulations, time_horizon) containing
            portfolio values for each simulation path over time.
        """
        returns = self.data.pct_change(fill_method=None).dropna(how='all').fillna(0)
        # Lasketaan geometrinen keskiarvo (CAGR) ja muutetaan se päivittäiseksi tuotto-odotukseksi
        total_growth = (1 + returns).prod()
        n_days = len(returns)
        geometric_mean_daily = total_growth**(1/n_days) - 1

        # Käytä tätä simulaatiossa
        mean_returns = geometric_mean_daily.values
        cov_matrix = returns.cov().values

        results = np.zeros((self.num_simulations, self.time_horizon))
        
        for i in range(self.num_simulations):
            # Generate correlated random returns for all assets
            sim_returns = np.random.multivariate_normal(mean_returns, cov_matrix, self.time_horizon)
            
            # Calculate weighted portfolio returns at each time step
            portfolio_returns = sim_returns @ self.weights
            
            # Track cumulative portfolio value over time
            cumulative_returns = np.cumprod(1 + portfolio_returns)
            results[i, :] = self.initial_investment * cumulative_returns
        
        self._results = results
        return results

    def get_statistics(self, percentiles=[5, 25, 50, 75, 95]):
        """
        Calculate statistics across all simulation paths.
        
        Parameters:
        -----------
        percentiles : list
            Percentiles to calculate (default: 5, 25, 50, 75, 95)
            
        Returns:
        --------
        dict
            Dictionary containing:
            - 'percentiles': dict of percentile values at each time step
            - 'mean': mean portfolio value at each time step
            - 'std': standard deviation at each time step
            - 'final_values': summary statistics for final portfolio values
        """
        if self._results is None:
            self.simulate()
        
        results = self._results
        final_values = results[:, -1]
        
        stats = {
            'percentiles': {p: np.percentile(results, p, axis=0) for p in percentiles},
            'mean': np.mean(results, axis=0),
            'std': np.std(results, axis=0),
            'final_values': {
                'mean': np.mean(final_values),
                'median': np.median(final_values),
                'std': np.std(final_values),
                'min': np.min(final_values),
                'max': np.max(final_values),
                'percentile_5': np.percentile(final_values, 5),
                'percentile_95': np.percentile(final_values, 95),
                'prob_loss': np.mean(final_values < self.initial_investment) * 100
            }
        }
        return stats

    def plot_simulation(self, show_percentiles=True, num_paths=100):
        """
        Plot Monte Carlo simulation results with percentile bands.
        
        Parameters:
        -----------
        show_percentiles : bool
            Whether to show percentile bands (5th, 50th, 95th)
        num_paths : int
            Number of individual paths to plot (for visual clarity)
        """
        if self._results is None:
            self.simulate()
        
        results = self._results
        stats = self.get_statistics()
        
        plt.figure(figsize=(12, 7))
        
        # Plot a subset of individual paths (semi-transparent)
        paths_to_plot = min(num_paths, self.num_simulations)
        for i in range(paths_to_plot):
            plt.plot(results[i, :], color='lightblue', alpha=0.3, linewidth=0.5)
        
        if show_percentiles:
            days = np.arange(self.time_horizon)
            
            # Plot percentile bands
            plt.fill_between(days, 
                           stats['percentiles'][5], 
                           stats['percentiles'][95],
                           color='blue', alpha=0.2, label='5th-95th percentile')
            plt.fill_between(days,
                           stats['percentiles'][25],
                           stats['percentiles'][75],
                           color='blue', alpha=0.3, label='25th-75th percentile')
            
            # Plot median
            plt.plot(stats['percentiles'][50], color='darkblue', 
                    linewidth=2, label='Median (50th percentile)')
        
        # Plot initial investment line
        plt.axhline(y=self.initial_investment, color='red', 
                   linestyle='--', linewidth=1, label=f'Initial: ${self.initial_investment:,.0f}')
        
        plt.title(f'Monte Carlo Simulation ({self.num_simulations:,} paths, {self.time_horizon} days)')
        plt.xlabel('Trading Days')
        plt.ylabel('Portfolio Value ($)')
        plt.legend(loc='upper left')
        plt.grid(True, alpha=0.3)
        
        # Add summary statistics text box
        final_stats = stats['final_values']
        textstr = f"Final Value Statistics:\n"
        textstr += f"Median: ${final_stats['median']:,.0f}\n"
        textstr += f"5th %%: ${final_stats['percentile_5']:,.0f}\n"
        textstr += f"95th %%: ${final_stats['percentile_95']:,.0f}\n"
        textstr += f"Prob. of Loss: {final_stats['prob_loss']:.1f}%"
        
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        plt.text(0.98, 0.02, textstr, transform=plt.gca().transAxes, 
                fontsize=9, verticalalignment='bottom', horizontalalignment='right',
                bbox=props)
        
        plt.tight_layout()
        plt.show()
        
    def print_summary(self):
        """Print a summary of simulation results."""
        if self._results is None:
            self.simulate()
            
        stats = self.get_statistics()
        final = stats['final_values']
        
        print("\n" + "="*50)
        print("MONTE CARLO SIMULATION SUMMARY")
        print("="*50)
        print(f"Initial Investment: ${self.initial_investment:,.0f}")
        print(f"Time Horizon: {self.time_horizon} trading days")
        print(f"Number of Simulations: {self.num_simulations:,}")
        print("-"*50)
        print("Final Portfolio Value Statistics:")
        print(f"  Mean:     ${final['mean']:,.0f}")
        print(f"  Median:   ${final['median']:,.0f}")
        print(f"  Std Dev:  ${final['std']:,.0f}")
        print(f"  Min:      ${final['min']:,.0f}")
        print(f"  Max:      ${final['max']:,.0f}")
        print("-"*50)
        print("Percentile Projections:")
        print(f"  5th percentile:  ${final['percentile_5']:,.0f}")
        print(f"  95th percentile: ${final['percentile_95']:,.0f}")
        print("-"*50)
        print(f"Probability of Loss: {final['prob_loss']:.1f}%")
        print("="*50)

### Run Monte Carlo Simulation

Uncomment to debug.

In [ ]:
# Monte Carlo simulation - now passing weights!
mc_sim = MonteCarloSimulation(data, weights=weights, num_simulations=1000, time_horizon=252)
mc_sim.print_summary()
mc_sim.plot_simulation()